# AI Cyber Defense Challenge — Nivelul 2 Baseline
## Clasificare Multi-clasă (10 Categorii de atac)

Baseline cu `LogisticRegression` pentru clasificarea celor 10 tipuri de trafic și atacuri.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

### 1. Încărcarea datelor (`level2_train.csv` și `level2_test.csv`)

In [ ]:
train_df = pd.read_csv("level2_train.csv")
test_df = pd.read_csv("level2_test.csv")

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

### 2. Codificarea etichetelor text (`attack_cat`)

In [ ]:
le = LabelEncoder()
y = le.fit_transform(train_df['attack_cat'])

X = train_df.drop(columns=['attack_cat'])
feature_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
X = X[feature_cols].fillna(0)

test_ids = test_df['id']
X_test = test_df[feature_cols].fillna(0)

print("Clase identificate:", list(le.classes_))

### 3. Validare cu Macro F1-Score

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
macro_f1 = f1_score(y_val, val_preds, average="macro", zero_division=0)

print(f"Macro F1-Score validare: {macro_f1:.4f}\n")
print(classification_report(y_val, val_preds, target_names=le.classes_, zero_division=0))

### 4. Generare submisie `level2_submission.csv`

In [ ]:
model.fit(X, y)
test_preds_encoded = model.predict(X_test)
test_preds = le.inverse_transform(test_preds_encoded)

submission = pd.DataFrame({
    "id": test_ids,
    "attack_cat": test_preds
})

submission.to_csv("level2_submission.csv", index=False)
print(f"Fisier generat: level2_submission.csv ({len(submission)} randuri)")
print(submission.head())